In [27]:
import time
import torch
import pandas as pd
import difflib
import json
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import Counter

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_NAME = "hynky/czech_news_dataset_v2"
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_PATH = "distilled-qwen-student"

In [3]:
login()

### Load dataset

In [ ]:
dataset = load_dataset(DATASET_NAME, split="train[:]")

### Load model

In [23]:
def get_model_dtype():
    return torch.float16 if DEVICE == "cuda" else torch.float32

model_dtype = get_model_dtype()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=model_dtype,
    device_map="auto",
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [24]:
def generate_summary(model, tokenizer, article):
    messages = [
        {
            "role": "system",
            "content": ("You are a professional Czech journalist and editor. Your task is to create a concise, accurate, and neutral summary of the provided newspaper article ONLY IN GRAMMATICALLY CORRECT CZECH. The summary should contain the most important facts and MUST NOT invent any new information, assumptions, or opinions. Be brief and factual."),
        },
        {
            "role": "user",
            "content": (
                "Summarize the following Czech article in 5-6 sentences."
                f"ČLÁNEK:\n{article}"
            ),
        },
    ]

    device = next(model.parameters()).device

    if hasattr(tokenizer, "apply_chat_template"):
        model_inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )

        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
        prompt_length = model_inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                **model_inputs,
                max_new_tokens=120,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_ids = output_ids[0][prompt_length:]
        return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    else:
        raise ValueError("Tokenizer does not support chat template")


def prepare_evaluation_data(dataset, model, tokenizer):
    eval_data = []

    start_time = time.time()
    data_len = len(dataset)

    for idx, item in enumerate(dataset, 1):
        article = item["content"]
        reference = item.get("brief", "")

        if (reference is None) or (reference.strip() == ""):
            continue

        summary = generate_summary(model, tokenizer, article)

        print("Summary:", summary)
        print("Ground truth:", reference)

        eval_data.append(
            {
                "response": summary,
                "reference": reference,
            }
        )

        print(f"[{idx} / {data_len}] Prepared example")

    elapsed_time = time.time() - start_time
    print(f"Finished preparing {data_len} examples in {elapsed_time:.2f}s")
    return eval_data

In [25]:
print("Generating summaries...")
t = time.time()
eval_data = prepare_evaluation_data(dataset, model, tokenizer)
print(f"Generated summaries in {(time.time() - t):.2f}s")

Generating summaries...
Summary: Skleněné koulesy a ohňosti zpomalily přibližným pohybovému procesem v Newyorském centrálním oblasti a Washingtonu, při kterém se objevila ohňosmělá vlny. Vířícé konfety a zápasy se objevil na oběma místnostech. V Washingtonu prezident Clinton předal svého manželu Hillary Clintonovi svůj titul prezidente Společenství
Ground truth: Ohňostroji, veselicemi a s jásotem přivítal svět příchod roku 2000. Jako poslední obyvatelé Země sledovali západ Slunce v roce 1999 obyvatelé tichomořského zámořského území USA, Americké Samoy. Slunce se naposledy v roce 1999 schovalo za duhovým horizontem v pátek 06:57 večer místního času (tj. v sobotu v 07:57 SEČ). Petardy a bengálské ohně usmrtily v Evropě šest lidí a desítky dalších zranily. Oslavy i přesto nepotvrdily chmurné představy policistů, kteří čekali daleko větší potíže a byli ve zvýšené pohotovosti ve všech evropských metropolích
[1 / 5] Prepared example
Summary: Nováčci Slavě a Bohemianů přijely do edenské rodin

In [26]:
df = pd.DataFrame(eval_data)
df.to_json(
    "generated_summaries.json",
    orient="records",
    indent=4,
    force_ascii=False,
)

### Rouge evaluation implementation

In [36]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def rouge_n(ref_text: str, out_text: str, n: int = 1):
    ref_tokens = ref_text.lower().split()
    cand_tokens = out_text.lower().split()

    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))

    overlap = sum((ref_ngrams & cand_ngrams).values())
    ref_count = sum(ref_ngrams.values())
    out_count = sum(cand_ngrams.values())

    recall = overlap / ref_count
    precision = overlap / out_count

    f1 = 0

    if (recall + precision > 0):
        f1 = (2 * recall * precision / (recall + precision))

    return {
        "overlap": overlap,
        "recall": recall,
        "precision": precision,
        "f1": f1,
    }

def lcs_length(x_tokens, y_tokens):
    matcher = difflib.SequenceMatcher(None, x_tokens, y_tokens)
    match = matcher.find_longest_match(0, len(x_tokens), 0, len(y_tokens))
    return match.size

def rouge_l(ref_text: str, out_text: str):
    ref_tokens = ref_text.lower().split()
    cand_tokens = out_text.lower().split()

    lcs = lcs_length(ref_tokens, cand_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(cand_tokens)

    f1 = 0
    if (recall + precision > 0):
      f1 = (2 * recall * precision / (recall + precision))

    return {
        "lcs": lcs,
        "recall": recall,
        "precision": precision,
        "f1": f1,
    }

In [31]:
def combine_baseline_student_summaries(student_file, baseline_file):
    with open(student_file, "r", encoding="utf-8") as f:
        student_data = json.load(f)

    with open(baseline_file, "r", encoding="utf-8") as f:
        baseline_data = json.load(f)

    combined_data = []

    for student_item, base_item in zip(student_data, baseline_data):
        if (base_item["reference"] != student_item["reference"]):
            continue

        combined_data.append({
            "response": student_item["response"],
            "reference": base_item["response"],
        })

    return combined_data

combined_eval_data = combine_baseline_student_summaries("generated_summaries_student.json", "generated_summaries_qwen1.5b.json")
df = pd.DataFrame(combined_eval_data)
df.to_json(
    "generated_summaries_student_x_qwen1.5b.json",
    orient="records",
    indent=4,
    force_ascii=False,
)

In [46]:
def evaluate_summaries(summaries):
    eval_data = []

    metrics = ["recall", "precision", "f1"]
    rouge_types = ["rouge-1", "rouge-2", "rouge-l"]
    totals = {rt: {m: 0.0 for m in metrics} for rt in rouge_types}

    with open(summaries, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        response = item.get("response")
        reference = item.get("reference")
        scores = {
                    "rouge-1": rouge_n(reference, response, n=1),
                    "rouge-2": rouge_n(reference, response, n=2),
                    "rouge-l": rouge_l(reference, response)
                }
        
        for rt in rouge_types:
            for m in metrics:
                totals[rt][m] += scores[rt][m]

        out = {
            "response": response,
            "reference": reference,
            **scores
            }
        
        eval_data.append(out)

    avg_obj = {rt: {m: (totals[rt][m] / len(eval_data)) for m in metrics} for rt in rouge_types}

    return eval_data, avg_obj

def save_evaluation_results(eval_data, average_scores, filename):
    output = {"eval_data": eval_data, "average_scores": average_scores}

    with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=4, ensure_ascii=False)

qwen_baseline_eval = evaluate_summaries("generated_summaries_qwen1.5b.json")
student_eval = evaluate_summaries("generated_summaries_student.json")
student_x_baseline_eval = evaluate_summaries("generated_summaries_student_x_qwen1.5b.json")

save_evaluation_results(qwen_baseline_eval[0], qwen_baseline_eval[1], "evaluations/qwen_baseline_eval.json")
save_evaluation_results(student_eval[0], student_eval[1], "evaluations/student_eval.json")
save_evaluation_results(student_x_baseline_eval[0], student_x_baseline_eval[1], "evaluations/student_x_baseline_eval.json")